# Мультимодальное слияние: фото плюс ингредиенты

Регрессия тех же четырех нутриентов, что и в baseline, но к визуальному
сигналу добавляется текстовый — список ингредиентов блюда. Сравниваем
три стратегии слияния (text-only, конкатенация, gated) с визуальным
baseline из ноутбука 02 на одних и тех же сплитах, целях и метриках.
Визуальные эмбеддинги переиспользуем без пересчета — это экономит GPU
и гарантирует прямую сопоставимость с baseline. Точечные предсказания
лучшей модели на calibration и test сохраняем для конформной калибровки
в ноутбуке 04.

## 1. Подготовка окружения

Подключены два input-датасета: `nutrition5k-overhead-rgb-224` (метаданные
и сплиты) и `nutrition5k-visual-baseline` (визуальные эмбеддинги, нормализация
целей, метрики baseline). Текстовый энкодер тянем с Hugging Face — нужен
Internet on на первом запуске.

In [ ]:
import json
import math
import os
import random
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

INPUT_DATA = Path("/kaggle/input/datasets/dbkarashev/nutrition5k-overhead-rgb-224/n5k_overhead")
INPUT_BASELINE = Path("/kaggle/input/datasets/dbkarashev/nutrition5k-visual-baseline/visual_baseline")

WORK_DIR = Path("/kaggle/working/multimodal_fusion")
TEXT_EMB_DIR = WORK_DIR / "text_embeddings"
PRED_DIR = WORK_DIR / "predictions"
MODELS_DIR = WORK_DIR / "models"
EDA_DIR = WORK_DIR / "eda"
for d in (WORK_DIR, TEXT_EMB_DIR, PRED_DIR, MODELS_DIR, EDA_DIR):
    d.mkdir(parents=True, exist_ok=True)

METRICS_FILE = WORK_DIR / "metrics.json"
STATUS_FILE = WORK_DIR / "_status.json"

TEXT_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TEXT_DIM = 384
VIS_DIM = 384
TARGETS = ["total_calories", "total_fat", "total_carb", "total_protein"]
RANDOM_SEED = 42

HEAD_HIDDEN = 256
HEAD_DROPOUT = 0.2
HEAD_BATCH = 256
HEAD_EPOCHS = 60
HEAD_LR = 1e-3
HEAD_WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 10


def set_seed(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")


def update_status(patch: dict) -> None:
    state = {}
    if STATUS_FILE.exists():
        try:
            state = json.loads(STATUS_FILE.read_text())
        except json.JSONDecodeError:
            pass
    state.update(patch)
    state["last_updated"] = now_iso()
    STATUS_FILE.write_text(json.dumps(state, indent=2, ensure_ascii=False))


set_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"INPUT_DATA:     {INPUT_DATA}")
print(f"INPUT_BASELINE: {INPUT_BASELINE}")
print(f"device:         {device}")
print(f"text encoder:   {TEXT_ENCODER_NAME}, dim={TEXT_DIM}")

## 2. Загрузка метаданных, сплитов, визуальных эмбеддингов и нормализации

Из baseline-датасета берем все три файла эмбеддингов и параметры нормализации
целей. Это означает, что обучение здесь использует ровно ту же визуальную
репрезентацию, что и baseline, а единственная разница — наличие текстового
канала и стратегия его слияния.

In [ ]:
dishes_df = pd.read_parquet(INPUT_DATA / "metadata" / "dishes.parquet")


def read_split(name: str) -> list[str]:
    return [l.strip() for l in (INPUT_DATA / "splits" / f"{name}.txt").read_text().splitlines() if l.strip()]


splits = {name: read_split(name) for name in ("train", "calibration", "test")}


def load_visual_embeddings(name: str) -> tuple[np.ndarray, list[str]]:
    data = np.load(INPUT_BASELINE / "embeddings" / f"{name}.npz", allow_pickle=False)
    return data["embeddings"], list(data["dish_ids"])


visual = {name: load_visual_embeddings(name) for name in ("train", "calibration", "test")}

norm = json.loads((INPUT_BASELINE / "target_norm.json").read_text())
mean = np.array(norm["mean"], dtype=np.float32)
std = np.array(norm["std"], dtype=np.float32)

baseline_metrics = json.loads((INPUT_BASELINE / "metrics.json").read_text())

for name, (emb, ids) in visual.items():
    print(f"  visual[{name:<11}] shape={emb.shape}  ids={len(ids)}")
print(f"  норм. mean: {dict(zip(TARGETS, [round(x, 2) for x in mean]))}")

## 3. Извлечение текстовых эмбеддингов

Кодируем ровно те строки, что лежат в `ingredient_names_str` (имена через
запятую, нижний регистр, без подчеркиваний — нормализация сделана в ноутбуке 01).
Кодируем строку целиком одним предложением, а не покомпонентно — модели полезен
контекст соседних ингредиентов. Эмбеддинги кешируются на диск; пересчитываются
только если файл отсутствует или его размер не сходится со списком dish_id.

In [ ]:
def text_emb_path(name: str) -> Path:
    return TEXT_EMB_DIR / f"{name}.npz"


def needs_text_extraction(name: str) -> bool:
    p = text_emb_path(name)
    if not p.exists():
        return True
    cached = np.load(p, allow_pickle=False)
    return len(cached["dish_ids"]) != len(visual[name][1])


splits_to_extract = [n for n in ("train", "calibration", "test") if needs_text_extraction(n)]

if not splits_to_extract:
    print("[STEP-3] Уже выполнен, пропускаю")
else:
    print(f"[STEP-3] Запуск: считаю текстовые эмбеддинги для {splits_to_extract}")
    from sentence_transformers import SentenceTransformer
    sbert = SentenceTransformer(TEXT_ENCODER_NAME, device=str(device))
    text_lookup = dishes_df.set_index("dish_id")["ingredient_names_str"].to_dict()

    for name in splits_to_extract:
        ids = visual[name][1]
        texts = [text_lookup[d] for d in ids]
        embs = sbert.encode(
            texts,
            batch_size=64,
            convert_to_numpy=True,
            normalize_embeddings=False,
            show_progress_bar=True,
        ).astype(np.float32)
        np.savez(text_emb_path(name), embeddings=embs, dish_ids=np.array(ids))

    del sbert
    if device.type == "cuda":
        torch.cuda.empty_cache()


def load_text_embeddings(name: str) -> tuple[np.ndarray, list[str]]:
    data = np.load(text_emb_path(name), allow_pickle=False)
    return data["embeddings"], list(data["dish_ids"])


text = {name: load_text_embeddings(name) for name in ("train", "calibration", "test")}
for name, (emb, ids) in text.items():
    assert ids == visual[name][1], f"text/visual ids mismatch на {name}"
    print(f"  text[{name:<11}]   shape={emb.shape}")

update_status({"step_3_text_embeddings_extracted": True})

## 4. Архитектуры слияния

Три модели поверх замороженных эмбеддингов. **TextOnly** — MLP только по
тексту, чтобы понять, сколько информации о нутриентах содержится уже в
одном лишь списке ингредиентов. **Concat** — простая конкатенация
`[v || t]` и одна MLP сверху, классическая ранняя fusion. **Gated** —
проектируем v и t в общее пространство, учим скалярный гейт по
конкатенации проекций и формируем взвешенную смесь, чтобы модель сама
решала, на какой канал опираться. Все головы одинаковой ширины и одинаковых
регуляризаций — это даст честное сравнение стратегий.

In [ ]:
def mlp(in_dim: int, hidden: int, out_dim: int, dropout: float) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(in_dim, hidden),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(hidden, hidden),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(hidden, out_dim),
    )


class TextOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = mlp(TEXT_DIM, HEAD_HIDDEN, len(TARGETS), HEAD_DROPOUT)

    def forward(self, v, t):
        return self.net(t)


class Concat(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = mlp(VIS_DIM + TEXT_DIM, HEAD_HIDDEN, len(TARGETS), HEAD_DROPOUT)

    def forward(self, v, t):
        return self.net(torch.cat([v, t], dim=-1))


class Gated(nn.Module):
    def __init__(self):
        super().__init__()
        self.v_proj = nn.Linear(VIS_DIM, HEAD_HIDDEN)
        self.t_proj = nn.Linear(TEXT_DIM, HEAD_HIDDEN)
        self.gate = nn.Sequential(
            nn.Linear(2 * HEAD_HIDDEN, HEAD_HIDDEN),
            nn.Sigmoid(),
        )
        self.head = mlp(HEAD_HIDDEN, HEAD_HIDDEN, len(TARGETS), HEAD_DROPOUT)

    def forward(self, v, t):
        v_h = self.v_proj(v)
        t_h = self.t_proj(t)
        g = self.gate(torch.cat([v_h, t_h], dim=-1))
        fused = g * v_h + (1.0 - g) * t_h
        return self.head(fused)


MODEL_CLASSES = {
    "text_only": TextOnly,
    "concat": Concat,
    "gated": Gated,
}

for name, cls in MODEL_CLASSES.items():
    n_params = sum(p.numel() for p in cls().parameters())
    print(f"  {name:<11}{n_params:>9,d} параметров")

## 5. Обучение

Один цикл на все три модели — гиперпараметры одинаковые, разнятся только
архитектуры. Лосс — Smooth L1 на нормализованных целях. AdamW + косинусное
расписание, ранняя остановка по validation loss на calibration. Чекпоинт
лучшей эпохи и история лоссов сохраняются под именем модели.

In [ ]:
def make_loader(name: str, shuffle: bool) -> DataLoader:
    v_emb, ids = visual[name]
    t_emb, _ = text[name]
    sub = dishes_df.set_index("dish_id").loc[ids]
    y = sub[TARGETS].to_numpy(dtype=np.float32)
    y_norm = (y - mean) / std
    ds = TensorDataset(
        torch.from_numpy(v_emb),
        torch.from_numpy(t_emb),
        torch.from_numpy(y_norm),
    )
    return DataLoader(ds, batch_size=HEAD_BATCH, shuffle=shuffle, drop_last=False)


def train_model(name: str) -> tuple[nn.Module, dict]:
    ckpt = MODELS_DIR / f"{name}.pt"
    history_path = MODELS_DIR / f"{name}_history.json"
    model = MODEL_CLASSES[name]().to(device)

    if ckpt.exists() and history_path.exists():
        print(f"[STEP-5] {name}: уже обучен, загружаю чекпоинт")
        model.load_state_dict(torch.load(ckpt, map_location=device))
        return model, json.loads(history_path.read_text())

    print(f"[STEP-5] {name}: запуск обучения")
    set_seed(RANDOM_SEED)
    model = MODEL_CLASSES[name]().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=HEAD_LR, weight_decay=HEAD_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=HEAD_EPOCHS)
    loss_fn = nn.SmoothL1Loss()

    train_loader = make_loader("train", shuffle=True)
    val_loader = make_loader("calibration", shuffle=False)

    history = {"train_loss": [], "val_loss": []}
    best_val = math.inf
    bad_epochs = 0
    best_state = None

    for epoch in range(1, HEAD_EPOCHS + 1):
        model.train()
        train_losses = []
        for vb, tb, yb in train_loader:
            vb = vb.to(device); tb = tb.to(device); yb = yb.to(device)
            pred = model(vb, tb)
            loss = loss_fn(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        scheduler.step()

        model.eval()
        with torch.inference_mode():
            val_losses = []
            for vb, tb, yb in val_loader:
                vb = vb.to(device); tb = tb.to(device); yb = yb.to(device)
                val_losses.append(loss_fn(model(vb, tb), yb).item())

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        improved = val_loss < best_val - 1e-5
        if improved:
            best_val = val_loss
            bad_epochs = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
        if bad_epochs >= EARLY_STOP_PATIENCE:
            print(f"  ранняя остановка на эпохе {epoch}, best val {best_val:.4f}")
            break

    model.load_state_dict(best_state)
    torch.save(best_state, ckpt)
    history_path.write_text(json.dumps(history, indent=2))
    return model, history


trained = {name: train_model(name) for name in MODEL_CLASSES}
update_status({"step_5_models_trained": list(MODEL_CLASSES.keys())})

## 6. Оценка и сохранение предсказаний

Для каждой модели считаем MAE и R^2 на test и calibration в исходных
единицах. MAPE специально не считаем — у части блюд `total_fat` или
`total_carb` около нуля, и MAPE получается мусорным. Точечные предсказания
каждой модели сохраняем в `predictions/<model>/{calibration,test}.parquet`.

In [ ]:
@torch.inference_mode()
def predict(model: nn.Module, name: str) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    v_emb, ids = visual[name]
    t_emb, _ = text[name]
    sub = dishes_df.set_index("dish_id").loc[ids]
    y_true = sub[TARGETS].to_numpy(dtype=np.float32)
    model.eval()
    pred_norm = model(torch.from_numpy(v_emb).to(device), torch.from_numpy(t_emb).to(device)).cpu().numpy()
    pred = pred_norm * std + mean
    df = pd.DataFrame({"dish_id": ids})
    for i, t in enumerate(TARGETS):
        df[f"pred_{t}"] = pred[:, i]
        df[f"true_{t}"] = y_true[:, i]
    return df, y_true, pred


def per_target_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    out = {}
    for i, t in enumerate(TARGETS):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        mae = float(np.mean(np.abs(yp - yt)))
        ss_res = float(np.sum((yp - yt) ** 2))
        ss_tot = float(np.sum((yt - yt.mean()) ** 2))
        r2 = 1.0 - ss_res / max(ss_tot, 1e-9)
        out[t] = {"mae": mae, "r2": r2}
    return out


metrics: dict = {"models": {}}
for name, (model, _) in trained.items():
    out_dir = PRED_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)
    cal_df, cal_y, cal_p = predict(model, "calibration")
    test_df, test_y, test_p = predict(model, "test")
    cal_df.to_parquet(out_dir / "calibration.parquet", index=False)
    test_df.to_parquet(out_dir / "test.parquet", index=False)
    metrics["models"][name] = {
        "calibration": per_target_metrics(cal_y, cal_p),
        "test": per_target_metrics(test_y, test_p),
    }

metrics["baseline_visual_only"] = {
    "calibration": {t: {"mae": baseline_metrics["calibration"][t]["mae"],
                        "r2": baseline_metrics["calibration"][t]["r2"]} for t in TARGETS},
    "test": {t: {"mae": baseline_metrics["test"][t]["mae"],
                 "r2": baseline_metrics["test"][t]["r2"]} for t in TARGETS},
}
METRICS_FILE.write_text(json.dumps(metrics, indent=2, ensure_ascii=False))

print("Test MAE:")
header = f"  {'target':<18}" + "".join(f"{n:>14}" for n in ["visual_only", *MODEL_CLASSES])
print(header)
for t in TARGETS:
    line = f"  {t:<18}"
    line += f"{metrics['baseline_visual_only']['test'][t]['mae']:>14.2f}"
    for name in MODEL_CLASSES:
        line += f"{metrics['models'][name]['test'][t]['mae']:>14.2f}"
    print(line)

print("\nTest R^2:")
print(header)
for t in TARGETS:
    line = f"  {t:<18}"
    line += f"{metrics['baseline_visual_only']['test'][t]['r2']:>14.3f}"
    for name in MODEL_CLASSES:
        line += f"{metrics['models'][name]['test'][t]['r2']:>14.3f}"
    print(line)

update_status({"step_6_evaluated": True})

## 7. Сравнительные графики

Слева — MAE по нутриентам с группировкой по моделям; справа — R^2.
Дополнительно сохраняем training_curves.png с тремя парами кривых.

In [ ]:
sns.set_theme(style="whitegrid", context="talk")
all_models = ["visual_only", *MODEL_CLASSES.keys()]
colors = {"visual_only": "gray", "text_only": "darkorange", "concat": "steelblue", "gated": "seagreen"}


def metric_at(name: str, target: str, key: str) -> float:
    if name == "visual_only":
        return metrics["baseline_visual_only"]["test"][target][key]
    return metrics["models"][name]["test"][target][key]


fig, axes = plt.subplots(1, 2, figsize=(15, 6))
x = np.arange(len(TARGETS))
width = 0.2
for i, name in enumerate(all_models):
    vals = [metric_at(name, t, "mae") for t in TARGETS]
    axes[0].bar(x + i * width - 1.5 * width, vals, width, label=name, color=colors[name])
    vals = [metric_at(name, t, "r2") for t in TARGETS]
    axes[1].bar(x + i * width - 1.5 * width, vals, width, label=name, color=colors[name])
for ax, title, ylabel in zip(axes, ["MAE на test", "R^2 на test"], ["MAE (исходные единицы)", "R^2"]):
    ax.set_xticks(x)
    ax.set_xticklabels([t.replace("total_", "") for t in TARGETS])
    ax.set(title=title, ylabel=ylabel)
    ax.legend(fontsize=10)
fig.tight_layout()
fig.savefig(EDA_DIR / "comparison_test.png", dpi=120)
plt.close(fig)

fig, ax = plt.subplots(figsize=(11, 6))
for name, (_, history) in trained.items():
    ax.plot(history["train_loss"], color=colors[name], ls="-", label=f"{name} train", alpha=0.6)
    ax.plot(history["val_loss"], color=colors[name], ls="--", label=f"{name} val", alpha=0.9)
ax.set(xlabel="Эпоха", ylabel="Smooth L1 (норм.)", title="Кривые обучения fusion-моделей")
ax.legend(fontsize=10, ncol=2)
fig.tight_layout()
fig.savefig(EDA_DIR / "training_curves.png", dpi=120)
plt.close(fig)

print(f"  графики в {EDA_DIR}: {sorted(p.name for p in EDA_DIR.glob('*.png'))}")
update_status({"step_7_eda_completed": True})

## 8. Финальный лог состояния

Сохраняем сводку: какая модель лучше всего сработала по среднему MAE на test
(это и будет источник предсказаний для ноутбука 04). После Save Version
опубликовать output как датасет `nutrition5k-multimodal-fusion`.

In [ ]:
def avg_test_mae(name: str) -> float:
    return float(np.mean([metric_at(name, t, "mae") for t in TARGETS]))


ranking = sorted([(name, avg_test_mae(name)) for name in all_models], key=lambda x: x[1])
best_name, best_mae = ranking[0]


def dir_size_mb(p: Path) -> float:
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / (1024 * 1024)


summary = {
    "text_encoder": TEXT_ENCODER_NAME,
    "vis_dim": VIS_DIM,
    "text_dim": TEXT_DIM,
    "targets": TARGETS,
    "head": {
        "hidden": HEAD_HIDDEN,
        "dropout": HEAD_DROPOUT,
        "epochs_max": HEAD_EPOCHS,
        "early_stop_patience": EARLY_STOP_PATIENCE,
    },
    "ranking_test_mae_avg": [{"model": n, "avg_mae": round(v, 3)} for n, v in ranking],
    "best_model": best_name,
    "best_avg_test_mae": round(best_mae, 3),
    "test_metrics": {
        name: {t: metric_at(name, t, "mae") for t in TARGETS}
        for name in all_models
    },
    "counts": {n: len(splits[n]) for n in ("train", "calibration", "test")},
    "sizes_mb": {
        "text_embeddings": dir_size_mb(TEXT_EMB_DIR),
        "predictions": dir_size_mb(PRED_DIR),
        "models": dir_size_mb(MODELS_DIR),
        "eda": dir_size_mb(EDA_DIR),
    },
    "timestamp": now_iso(),
}
update_status({"step_8_summary_written": True, "summary": summary})

print(json.dumps(summary, indent=2, ensure_ascii=False))
print()
print(f"Лучшая модель по среднему MAE на test: {best_name}  (avg MAE={best_mae:.2f})")
print("Save & Run All (Commit), затем опубликуй output как nutrition5k-multimodal-fusion для ноутбука 04.")